# Pursuer RL training

End-to-end notebook: env → PPO → evaluation → ONNX export.

Runs in Google Colab (GPU recommended) or local Jupyter. Paired files live at
https://github.com/unclestep/Rogue/tree/develop/rl. Trained model goes to
`rl/models/pursuer.onnx`; Go loads it via `ROGUE_PURSUER_MODEL_PATH`.

**Do not edit the observation layout without updating
`internal/domain/service/rl_observation.go` in lockstep.** The parity cell at
the bottom catches divergence, but only after the fact.


## 1. Setup

In [1]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab:' , IN_COLAB)


Colab: False


In [11]:
if IN_COLAB:
    %pip install --quiet 'gymnasium>=1.0' 'stable-baselines3>=2.4' torch onnx onnxscript onnxruntime tensorboard matplotlib


## 2. Hyperparameters

In [3]:
from pathlib import Path

CONFIG = {
    'topologies_dir': 'fixtures/topologies',  # committed fixtures; swap for 'topologies' after make dump-topologies
    'total_timesteps': 2_000_000,   # bump to 2_000_000+ for real training
    'n_envs': 8,
    'max_episode_steps': 200,
    'seed': 42,
    'model_out': 'models/pursuer.onnx',
    'checkpoint_dir': 'checkpoints',
    'tensorboard_log': 'runs',
    # --- Domain randomization ---
    # Per-episode episode cap (lower bound training with short lives, upper
    # bound with long stalking). None = fixed max_episode_steps above.
    'max_episode_steps_range': (120, 300),
    # Probability of spawning a wandering distractor monster per episode.
    # Forces the policy to treat the `other_monster` channel as noise.
    'distractor_prob': 0.3,
    # Sample scripted-player profile per episode (default/aggressive/cautious/timid).
    'randomize_player_profile': True,
}
Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path('models').mkdir(parents=True, exist_ok=True)
CONFIG


{'topologies_dir': 'fixtures/topologies',
 'total_timesteps': 2000000,
 'n_envs': 8,
 'max_episode_steps': 200,
 'seed': 42,
 'model_out': 'models/pursuer.onnx',
 'checkpoint_dir': 'checkpoints',
 'tensorboard_log': 'runs',
 'max_episode_steps_range': (120, 300),
 'distractor_prob': 0.3,
 'randomize_player_profile': True}

## 3. Environment modules

Sync the env/player modules into the notebook's working dir. In Colab this
pulls fresh copies from the repo; locally you can symlink or rely on the
current working directory being `rl/`.

In [4]:
import os, pathlib, shutil

if IN_COLAB:
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/unclestep/Rogue.git _repo
    !cd _repo && git sparse-checkout set rl cmd/dump-topology
    for name in ('pursuer_env.py', 'scripted_player.py', '__init__.py'):
        src = pathlib.Path('_repo/rl') / name
        if src.exists():
            shutil.copy(src, name)
    shutil.copytree('_repo/rl/fixtures', 'fixtures', dirs_exist_ok=True)
else:
    # Local: assume we're running with cwd=rl/ (jupyter lab rl/pursuer_training.ipynb).
    pass

print('env files in cwd:', sorted(f for f in os.listdir('.') if f.endswith('.py')))


env files in cwd: ['__init__.py', 'pursuer_env.py', 'scripted_player.py']


In [5]:
# Import as local modules — both Colab-copied and local paths work.
import importlib, sys
sys.path.insert(0, '.')
import pursuer_env, scripted_player
importlib.reload(pursuer_env)
importlib.reload(scripted_player)
from pursuer_env import PursuerEnv, OBSERVATION_SIZE, ACTION_COUNT
from scripted_player import scripted_player_policy
print('observation_size =', OBSERVATION_SIZE, 'actions =', ACTION_COUNT)


observation_size = 859 actions = 5


## 4. Sanity check: single env

In [6]:
env = PursuerEnv(CONFIG['topologies_dir'], scripted_player_policy, max_episode_steps=50, seed=0)
obs, info = env.reset(seed=0)
print('obs', obs.shape, obs.dtype, 'range', obs.min(), obs.max())
print('loaded', info)
total = 0.0
for _ in range(30):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    total += r
    if term or trunc: break
print('30-step random rollout reward =', round(total, 3))


obs (859,) float32 range 0.0 1.0
loaded {'topology': '0012.json', 'player_profile': 'default', 'has_distractor': False, 'max_steps': 50}
30-step random rollout reward = -0.24


## 5. PPO training

In [8]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback

def make_env(rank: int):
    def _thunk():
        return PursuerEnv(
            CONFIG['topologies_dir'],
            scripted_player_policy,
            max_episode_steps=CONFIG['max_episode_steps'],
            max_episode_steps_range=CONFIG.get('max_episode_steps_range'),
            distractor_prob=CONFIG.get('distractor_prob', 0.0),
            randomize_player_profile=CONFIG.get('randomize_player_profile', False),
            seed=CONFIG['seed'] + rank,
        )
    return _thunk

vec = SubprocVecEnv([make_env(i) for i in range(CONFIG['n_envs'])])
vec = VecMonitor(vec)

model = PPO(
    'MlpPolicy',
    vec,
    device="cpu",
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=512,
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=dict(net_arch=[128, 128]),
    tensorboard_log=CONFIG['tensorboard_log'],
    seed=CONFIG['seed'],
    verbose=1,
)

ckpt_cb = CheckpointCallback(
    save_freq=max(10_000 // CONFIG['n_envs'], 1),
    save_path=CONFIG['checkpoint_dir'],
    name_prefix='pursuer',
)
model.learn(total_timesteps=CONFIG['total_timesteps'], callback=ckpt_cb)


Using cpu device
Logging to runs/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 197      |
|    ep_rew_mean     | -2.2     |
| time/              |          |
|    fps             | 2826     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 201         |
|    ep_rew_mean          | -2.06       |
| time/                   |             |
|    fps                  | 2651        |
|    iterations           | 2           |
|    time_elapsed         | 6           |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.015953168 |
|    clip_fraction        | 0.154       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | -0.85

## 6. Evaluation

In [9]:
import numpy as np

from pursuer_env import ACTION_VECTORS, ACTION_WAIT, chase_scent_map


class BaselinePolicy:
    """
    Greedy chase against the player's *true* position — same algorithm as Go's
    `dijkstraCardinal.findOptimals` (sys_pathfinder.go:128): the center cell
    only acts as a baseline for `minScent`, and any cardinal neighbor whose
    scent is ≤ center gets considered; among the tied-minimum candidates, Go
    picks randomly. Here we tie-break deterministically by Manhattan distance
    to the player so eval is reproducible.

    This is the skill floor: the RL policy must outperform it on stealth/
    ambush metrics, otherwise RL adds no value over hand-coded ChaseBehavior.
    It has *strictly better information* than the Pursuer — reads player_pos
    directly, no LOS or memory decay. That's intentional.
    """

    def __init__(self, env):
        self._env = env

    def predict(self, obs, deterministic=True):
        env = self._env
        assert env.topology is not None
        scent = chase_scent_map(env.topology, env.player_pos)
        ox, oy = env.pursuer_pos
        center_val = int(scent[oy, ox])

        candidates = []
        min_val = center_val
        for action, (dx, dy) in ACTION_VECTORS.items():
            if action == ACTION_WAIT:
                continue
            nx, ny = ox + dx, oy + dy
            if not env.topology.is_walkable(nx, ny):
                continue
            v = int(scent[ny, nx])
            if v < min_val:
                min_val = v
                candidates = [(action, nx, ny)]
            elif v == min_val:
                candidates.append((action, nx, ny))

        if not candidates:
            return ACTION_WAIT, None
        # Deterministic tie-break: cardinal with smallest Manhattan distance
        # to the player (Go randomises; we want reproducible eval runs).
        px, py = env.player_pos
        action, _, _ = min(
            candidates,
            key=lambda c: (abs(c[1] - px) + abs(c[2] - py), c[0]),
        )
        return action, None


def rollout(policy_name, policy, eval_env, n_episodes=30, seed_base=1000):
    returns, lengths, in_cone_ratios = [], [], []
    catches, total_hits, ambush_hits = 0, 0, 0
    first_hit_turns = []
    for ep in range(n_episodes):
        obs, _ = eval_env.reset(seed=seed_base + ep)
        ep_ret, ep_steps, ep_in_cone = 0.0, 0, 0
        ep_hits, ep_ambush = 0, 0
        first_hit = None
        done = False
        info = {}
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            obs, r, term, trunc, info = eval_env.step(int(action))
            ep_ret += r
            ep_steps += 1
            if info.get('in_cone'):
                ep_in_cone += 1
            if info.get('pursuer_hit'):
                ep_hits += 1
                if first_hit is None:
                    first_hit = ep_steps
            if info.get('ambush_hit'):
                ep_ambush += 1
            done = term or trunc
        returns.append(ep_ret)
        lengths.append(ep_steps)
        in_cone_ratios.append(ep_in_cone / max(ep_steps, 1))
        total_hits += ep_hits
        ambush_hits += ep_ambush
        if info.get('caught'):
            catches += 1
        if first_hit is not None:
            first_hit_turns.append(first_hit)
    return {
        'policy': policy_name,
        'avg_return': float(np.mean(returns)),
        'std_return': float(np.std(returns)),
        'catch_rate': catches / n_episodes,
        'avg_in_cone': float(np.mean(in_cone_ratios)),
        'avg_length': float(np.mean(lengths)),
        'avg_first_hit_turn': float(np.mean(first_hit_turns)) if first_hit_turns else float('nan'),
        'total_hits': total_hits,
        'ambush_hits': ambush_hits,
        'ambush_ratio': ambush_hits / max(total_hits, 1),
    }


eval_env = PursuerEnv(
    CONFIG['topologies_dir'],
    scripted_player_policy,
    max_episode_steps=CONFIG['max_episode_steps'],
    seed=999,
)

results = [
    rollout('PPO', model, eval_env, n_episodes=30),
    rollout('Baseline (chase-scent)', BaselinePolicy(eval_env), eval_env, n_episodes=30),
]

header = (
    f"{'policy':<24} {'return':>14} {'catch':>7} {'in_cone':>9} "
    f"{'len':>6} {'1st_hit':>8} {'hits':>6} {'ambush':>8}"
)
print(header)
print('-' * len(header))
for r in results:
    first_hit = (
        f"{r['avg_first_hit_turn']:>8.1f}"
        if r['avg_first_hit_turn'] == r['avg_first_hit_turn']
        else f"{'nan':>8}"
    )
    print(
        f"{r['policy']:<24} "
        f"{r['avg_return']:>7.2f} ± {r['std_return']:>4.2f} "
        f"{r['catch_rate']:>7.1%} "
        f"{r['avg_in_cone']:>9.3f} "
        f"{r['avg_length']:>6.1f} "
        f"{first_hit} "
        f"{r['total_hits']:>6d} "
        f"{r['ambush_ratio']:>8.1%}"
    )

# The RL policy wins this comparison when it lowers `in_cone` and lifts
# `ambush` without losing too much on `catch`/`1st_hit`. If PPO trails the
# baseline on every axis after a real training run, reward shaping needs
# tuning (see PR 5).


policy                           return   catch   in_cone    len  1st_hit   hits   ambush
-----------------------------------------------------------------------------------------
PPO                        29.81 ± 38.34   30.0%     0.080  155.9     88.3     60    46.7%
Baseline (chase-scent)     17.15 ± 24.74    6.7%     0.114   64.4     59.8     39    71.8%


## 7. ONNX export

Export a torch module that maps flat observation → raw logits, matching
Go's `ONNXPolicy.Predict` (argmax done in Go). Input name `input`, output
name `logits`, opset 17.

In [12]:
import torch
import torch.nn as nn

class PursuerActor(nn.Module):
    """Wraps the SB3 policy so torch.onnx.export sees a clean forward(obs) -> logits."""

    def __init__(self, sb3_policy):
        super().__init__()
        self.policy = sb3_policy

    def forward(self, obs):
        # SB3 >= 2.0 layout. Bump the check here if you upgrade SB3.
        features = self.policy.extract_features(obs, self.policy.pi_features_extractor)
        latent_pi, _ = self.policy.mlp_extractor(features)
        return self.policy.action_net(latent_pi)

actor = PursuerActor(model.policy).eval()
dummy = torch.zeros(1, OBSERVATION_SIZE)
torch.onnx.export(
    actor,
    dummy,
    CONFIG['model_out'],
    input_names=['input'],
    output_names=['logits'],
    opset_version=17,
    dynamic_axes=None,  # fixed shape — Go sends [1, 859] exactly
)
print('exported', CONFIG['model_out'])


W0418 02:51:19.030000 58578 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0418 02:51:19.372000 58578 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0418 02:51:19.373000 58578 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0418 02:51:19.373000 58578 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `PursuerActor([...]` with `torch.export.export(..., strict=False)`...


/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Obtain model graph for `PursuerActor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported models/pursuer.onnx


## 8. Parity check — PyTorch vs ONNX

In [13]:
import onnxruntime as ort

sess = ort.InferenceSession(CONFIG['model_out'])
rng = np.random.default_rng(0)
max_diff = 0.0
for _ in range(1000):
    sample = rng.uniform(-1, 1, size=(1, OBSERVATION_SIZE)).astype(np.float32)
    with torch.no_grad():
        py_logits = actor(torch.from_numpy(sample)).numpy()
    ort_logits = sess.run(['logits'], {'input': sample})[0]
    max_diff = max(max_diff, float(np.max(np.abs(py_logits - ort_logits))))
print(f'max |PyTorch - ONNX| = {max_diff:.2e}')
assert max_diff < 1e-4, 'ONNX export diverged from PyTorch'


max |PyTorch - ONNX| = 2.86e-06


## 9. Download (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(CONFIG['model_out'])
else:
    print('local run — model saved to', CONFIG['model_out'])
